# Blaze 核心接口速览

> **新手提示**：本节只介绍最常用的核心接口，帮助你快速上手。完整接口详见附录。

本节学习大纲：
- GemmUniversal——算子入口（一行代码执行矩阵乘）
- BlockMmad——计算组件（搬运→计算→输出）
- BlockScheduler——调度组件（切分矩阵、分配任务）
- BlockEpilogue——后处理组件（累加、激活等）
- Layout 类型——四种布局及新手推荐
---



## 0. 快速回顾：组件组装流程

在 [01.03](./01.03_blaze_programming_paradigm.ipynb) 我们学到了 Blaze 的组装范式：

```
选择策略 → 定义类型 → 组合组件 → 组装 Kernel → 构造参数 → 执行
```

本节详细介绍每个组件的核心接口，让你知道"每个组件需要什么参数"。



## 1. GemmUniversal——算子入口

### 1.1 它是什么？

**类比**：GemmUniversal 是矩阵乘算子的"主入口"，就像工厂的主管。
- 接收任务参数（矩阵维度、地址）
- 分配任务给工人（BlockScheduler 切分）
- 指挥工人干活（BlockMmad 计算）
- 检验结果（BlockEpilogue 后处理）

### 1.2 模板签名（需要 4 个组件）

```cpp
template <
    class ProblemShape_,    // 问题规模 (m, n, k, batch)
    class BlockMmad_,       // 计算组件
    class BlockEpilogue_,   // 后处理组件
    class BlockScheduler_   // 调度组件
> class GemmUniversal;
```

### 1.3 核心参数说明

<table style="text-align: left; margin-left: 0">
<tr><th align="left">参数</th><th align="left">是什么</th><th align="left">新手理解</th></tr>
<tr><td align="left">ProblemShape_</td><td align="left">Shape<int64_t, 4></td><td align="left">矩阵维度，例如 {1024, 1024, 512, 1}</td></tr>
<tr><td align="left">BlockMmad_</td><td align="left">BlockMmad<...></td><td align="left">实际干活的组件</td></tr>
<tr><td align="left">BlockEpilogue_</td><td align="left">Empty 或 StreamK</td><td align="left">Basic 场景用 Empty</td></tr>
<tr><td align="left">BlockScheduler_</td><td align="left">BlockScheduler...</td><td align="left">负责切分矩阵</td></tr>
</table>

### 1.4 执行方式（一行代码）

```cpp
Kernel mm;
mm(params);  // 一行代码完成矩阵乘！
```

### 1.5 Params 结构（传递什么参数）

```cpp
struct Params {
    ProblemShape problemShape;            // {m, n, k, batch}
    BlockMmad::Params mmadParams;         // 包含 A/B/C/Bias 的 GM 地址
    BlockEpilogue::Params epilogueParams; // 后处理参数（Basic 为空）
    BlockScheduler::Params schedulerParams; // 分块参数
};
```

**新手提示**：Basic 场景下，你只需要提供：
1. 矩阵维度 (m, n, k)
2. A/B/C/Bias 的内存地址
3. 分块参数（Host 侧计算，详见 01.05）



## 2. BlockMmad——计算组件

### 2.1 它是什么？

**类比**：BlockMmad 是"工人"，负责实际干活。
- 从 GM 搬运数据到 L1
- 从 L1 搬运数据到 L0
- 在 L0 执行 MMAD 计算
- 把结果写回 GM

### 2.2 模板签名（需要策略+类型+布局）

```cpp
template <
    class DispatchPolicy_,          // 策略（决定用什么流水线）
    class AType_, class LayoutA_,   // A 矩阵：类型 + 布局
    class BType_, class LayoutB_,   // B 矩阵：类型 + 布局
    class CType_, class LayoutC_,   // C 矩阵：类型 + 布局
    class BiasType_, class LayoutBias_ // Bias：类型 + 布局
> class BlockMmad;
```

### 2.3 核心参数说明

<table style="text-align: left; margin-left: 0">
<tr><th align="left">参数</th><th align="left">新手理解</th></tr>
<tr><td align="left">DispatchPolicy_</td><td align="left">策略：Basic 用 MatmulMultiBlockBasic<0, 0></td></tr>
<tr><td align="left">AType_</td><td align="left">数据类型：half / bfloat16_t / float 等</td></tr>
<tr><td align="left">LayoutA_</td><td align="left">布局：NDExtLayoutPtn（新手推荐）</td></tr>
<tr><td align="left">...</td><td align="left">B/C/Bias 同理</td></tr>
</table>

### 2.4 编译期自动推导

BlockMmad 会根据你传入的 Layout 自动推导：
- `transA` / `transB`：是否需要转置（通过 IsTrans 推导）
- `weightNZFormat`：B 是否是 WeightNZ 格式

**新手提示**：你不需要手动设置这些，Blaze 会自动处理！



## 3. BlockScheduler——调度组件

### 3.1 它是什么？

**类比**：BlockScheduler 是"调度员"，负责切蛋糕。
- 把大矩阵切成小块（Tile）
- 分配给不同的核去计算
- 每个核独立完成自己的 Block

### 3.2 模板签名

```cpp
template <class ProblemShape_, uint64_t FullLoadMode_ = 0>
class BlockScheduler;
```

### 3.3 核心接口

<table style="text-align: left; margin-left: 0">
<tr><th align="left">接口</th><th align="left">返回值</th><th align="left">新手理解</th></tr>
<tr><td align="left">GetTileNum()</td><td align="left">int64_t</td><td align="left">总共切成多少块</td></tr>
<tr><td align="left">GetBlockShape(idx)</td><td align="left">BlockShape</td><td align="left">第 idx 块的形状</td></tr>
<tr><td align="left">GetBlockCoord(idx)</td><td align="left">BlockCoord</td><td align="left">第 idx 块的坐标位置</td></tr>
</table>

### 3.4 Params 参数（Host 侧计算）

```cpp
struct Params {
    uint32_t mL1 = 0;      // L1 Tile 尺寸（搬运粒度）
    uint32_t nL1 = 0;
    uint32_t kL1 = 0;
    uint32_t baseM = 0;    // L0 Tile 尺寸（计算粒度）
    uint32_t baseN = 0;
    uint32_t baseK = 0;
    uint32_t mTailCnt = 0; // 尾块余量（处理不能整除的部分）
    uint32_t nTailCnt = 0;
    // ... 其他缓冲参数
};
```

**类比**：
- mL1：卡车一次运多少货（搬运粒度）
- baseM：工人一次处理多少货（计算粒度）

**新手提示**：这些参数在 Host 侧通过 Tiling 计算，详见 [01.05](./01.05_blaze_basic_matmul.ipynb)。



## 4. BlockEpilogue——后处理组件

### 4.1 它是什么？

**类比**：BlockEpilogue 是"质检员"，负责后处理。
- 累加中间结果（StreamK 场景）
- 类型转换（float → half）
- 激活函数（ReLU 等）

### 4.2 两种实现

<table style="text-align: left; margin-left: 0">
<tr><th align="left">实现</th><th align="left">适用场景</th><th align="left">新手理解</th></tr>
<tr><td align="left">BlockEpilogueEmpty</td><td align="left">Basic 非量化</td><td align="left">不需要后处理，空实现</td></tr>
<tr><td align="left">BlockEpilogueStreamK</td><td align="left">StreamK</td><td align="left">累加 workspace 结果</td></tr>
</table>

### 4.3 Basic 场景为什么用 Empty？

**原因**：Basic 场景下，L0C 结果直接通过 ON_THE_FLY 写回 GM，不需要 AIV 端额外处理。

**使用方式**：
```cpp
using BlockEpilogue = Blaze::Gemm::Block::BlockEpilogueEmpty;
```

**新手提示**：入门阶段，Basic 非量化场景直接用 Empty 就对了！



## 5. Layout 类型——四种布局

### 5.1 什么是布局？

布局决定了数据在内存中的排列方式：
- **ND（Row-major）**：按行排列 [0,1,2,3,4,5...] ← 训练场景常见
- **NZ/ZN（分形）**：按块排列 ← 推理场景优化

### 5.2 四种布局对比

<table style="text-align: left; margin-left: 0">
<tr><th align="left">布局类型</th><th align="left">含义</th><th align="left">IsTrans</th><th align="left">IsWeightNz</th></tr>
<tr><td align="left">NDExtLayoutPtn</td><td align="left">ND 扩展布局（非转置）</td><td align="left">false</td><td align="left">false</td></tr>
<tr><td align="left">DNExtLayoutPtn</td><td align="left">DN 扩展布局（转置）</td><td align="left">true</td><td align="left">false</td></tr>
<tr><td align="left">NZLayoutPtn</td><td align="left">NZ 分形布局</td><td align="left">false</td><td align="left">true</td></tr>
<tr><td align="left">ZNLayoutPtn</td><td align="left">ZN 分形布局（转置）</td><td align="left">true</td><td align="left">true</td></tr>
</table>

### 5.3 使用推荐

**入门阶段，大部分情况用这个就够了**：
```cpp
using LayoutA = AscendC::Te::NDExtLayoutPtn;  // A: ND
using LayoutB = AscendC::Te::NDExtLayoutPtn;  // B: ND
using LayoutC = AscendC::Te::NDExtLayoutPtn;  // C: ND
```

**推理场景可能用**：
```cpp
using LayoutB = AscendC::Te::ZNLayoutPtn;  // B: 转置 + WeightNZ
```

---



## 6. 本节小结

### 新手需要记住的核心要点

1. **GemmUniversal**：算子入口，一行代码执行
2. **BlockMmad**：工人，负责搬运+计算
3. **BlockScheduler**：调度员，切分矩阵
4. **BlockEpilogueEmpty**：Basic 场景用这个
5. **Layout**：一般场景都用 NDExtLayoutPtn

### Basic 场景组装模板

```cpp
// 1. 选择策略
using Policy = Blaze::Gemm::MatmulMultiBlockBasic<0, 0>; // Basic 策略

// 2. 定义类型
using AType = bfloat16_t;  // BF16
using LayoutA = AscendC::Te::NDExtLayoutPtn;  // ND 布局

// 3. 组装 Kernel
using Kernel = Blaze::Gemm::Kernel::GemmUniversal<
    ProblemShape, BlockMmad, BlockEpilogueEmpty, BlockScheduler>; // Empty 后处理

// 4. 执行
Kernel mm;
mm(params);  // 完成！
```

> **下一步**：[01.05 Basic MatMul 实操](./01.05_blaze_basic_matmul.ipynb) 将上述模板完整落地。



## 7. 课后练习

1. **（判断题）** `GemmUniversal` 的 Params 结构在不同 Kernel 特化版本中完全相同。

2. **（判断题）** BlockMmad 会根据 Layout 自动推导 transA/transB，不需要手动设置。

3. **（单选题）** Basic 非量化场景应该用哪个 Epilogue？
   - A. BlockEpilogueStreamK
   - B. BlockEpilogueEmpty
   - C. DefaultFusion
   - D. 不需要 Epilogue

4. **（单选题）** `ZNLayoutPtn` 的 IsTrans 和 IsWeightNz 特征值分别为：
   - A. false, false
   - B. true, false
   - C. false, true
   - D. true, true

5. **（填空题）** BlockScheduler 的 `mL1/nL1/kL1` 控制 ______ 级搬运粒度，`baseM/baseN/baseK` 控制 ______ 级计算粒度。

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/01.04_answer.txt